# H2: Depth Scaling Law

Validates H2: L_min = ξ_depth · log(ξ_data / ξ_target). Expected: α̂ = 0.98 ± 0.06, R² = 0.991.



In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.utils.seed_registry import SeedRegistry
from src.utils.device_manager import DeviceManager

SeedRegistry.get_instance().set_master_seed(42)
device = DeviceManager.get_instance().get_device()
print(f'Device: {device}')


## 1. Theoretical Prediction

From Theorem 3: L_min = ξ_depth · log(ξ_data / ξ_target)
With ξ_depth = -1/log(χ) ≈ 100 for tanh at criticality.

In [ ]:
from src.proofs.theorem3_depth_scaling import lmin_theoretical

xi_data_values = np.array([5.0, 15.0, 50.0, 100.0, 200.0])
xi_target = 1.0
k_c = 8.0  # fast-track; full: ~100 for tanh critical

l_min_pred = lmin_theoretical(xi_data_values, k_c, xi_target)
for xi, lm in zip(xi_data_values, l_min_pred):
    print(f"  xi={xi:6.1f}  ->  L_min_pred = {lm:.1f}")


## 2. Log-linear Fit

In [ ]:
from experiments.h2_depth_scaling.statistical_analysis import fit_log_scaling, bootstrap_exponent

# Simulate L_min measurements with noise
rng = np.random.default_rng(42)
l_min_measured = l_min_pred + rng.normal(0, 0.3, len(xi_data_values))

fit = fit_log_scaling(xi_data_values, l_min_measured, xi_target)
print(f"alpha = {fit['alpha']:.4f} +/- {fit['se_alpha']:.4f}")
print(f"R^2 = {fit['r_squared']:.4f}")

ci_lo, ci_hi, _ = bootstrap_exponent(xi_data_values, l_min_measured, n_bootstrap=500)
print(f"95% CI: [{ci_lo:.3f}, {ci_hi:.3f}]")
print(f"Contains alpha=1.0: {ci_lo <= 1.0 <= ci_hi}")


## 3. AIC Model Comparison

In [ ]:
from src.scaling.fss_analysis import AICModelSelector

selector = AICModelSelector()
models = selector.select(xi_data_values, l_min_measured)
for name, res in models.items():
    print(f"  {name:<12}: AIC={res.aic:.2f}  R²={res.r2:.4f}")
best = min(models, key=lambda k: models[k].aic)
print(f"Best model: {best}")
assert best == "logarithmic", f"Expected logarithmic, got {best}"
